# 04 Full pipeline run: one config per session (Task 5 runner + Task 7 metrics)

This notebook runs **one config per session**, saves its result files, and computes the Task 7 outputs.

## How to choose the config
1. Go to the cell titled **"1. Choose the config"** and change `CONFIG_NAME`.
2. Run all cells from top to bottom.
3. To test another config: **Runtime > Restart session** (this frees GPU memory, one model at a time), change `CONFIG_NAME`, run again.
   Each run adds or replaces its own row in `results/metrics.csv`, so after running all five you get the full comparison table.

| Config | Embedder | Retriever | Reranker | LLM | Needs |
|---|---|---|---|---|---|
| `arch1` | e5 | dense | none | qwen3b | - |
| `arch2` | bge | dense | none | qwen7b | - |
| `arch3` | bge | hybrid | bge | aya8b | HF token + Aya license |
| `arch1_bge` | bge | dense | none | qwen3b | - |
| `arch3_no_rr` | bge | hybrid | none | aya8b | HF token + Aya license |

**Parallel runs:** each teammate runs a different config in their own session. Every session writes its own `metrics.csv` with one row, so send each zip to the person who runs notebook 05, which merges them.

Other names in `configs.py` (for example `arch2_qwen3`) also work. The cell below prints every available name.

**Needs a GPU**: Colab (Runtime > Change runtime type > T4 GPU) or Kaggle (Accelerator > GPU).

## 1. Choose the config

In [ ]:
# ============ CHANGE ONLY THIS LINE ============
CONFIG_NAME = "arch1"   # arch1 | arch2 | arch3 | arch1_bge | arch3_no_rr
# ===============================================

TOP_K = 5          # chunks given to the LLM (fixed by the spec)
CANDIDATES = 20    # candidates fetched before reranking (fixed by the spec)
SEED = 42

## 2. Setup (clone repo, install packages, set paths)

In [ ]:
import os, sys, json, time, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/yousefJooX/egyptian-personal-status-rag.git"
REPO_NAME = "egyptian-personal-status-rag"

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# locate (or clone) the repository
cwd = Path.cwd()
if (cwd / "run.py").exists() or (cwd / "demo").exists():
    ROOT = cwd
else:
    ROOT = cwd / REPO_NAME
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)

DEMO = ROOT / "demo" if (ROOT / "demo").exists() else ROOT
os.chdir(DEMO)                     # blocks cache their indexes in data/index relative to here
for p in (str(DEMO), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

# install only what is missing (torch and transformers already ship with Colab/Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "accelerate", "bitsandbytes", "faiss-cpu", "FlagEmbedding",
                "rank-bm25", "sentence-transformers", "pandas", "tqdm"], check=True)

print("Environment:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local")
print("ROOT:", ROOT, "| working dir:", Path.cwd())

In [ ]:
# Hugging Face token (only needed for gated models such as aya8b)
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
print("HF_TOKEN:", "loaded" if token else "not found (fine for qwen configs, required for aya8b)")

## 3. Load data and check the config

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from configs import CONFIGS
from registry import build

def find_file(name):
    for p in (DEMO / "data" / name, ROOT / "data" / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found in {DEMO/'data'} or {ROOT/'data'}")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

if CONFIG_NAME not in CONFIGS:
    raise ValueError(f"Unknown config '{CONFIG_NAME}'. Available: {list(CONFIGS)}")

chunks = load_json(find_file("chunks.json"))
test_set = load_json(find_file("test_set.json"))
chunks_by_id = {c["id"]: c for c in chunks}

# test set sanity checks
qids = [q["qid"] for q in test_set]
assert len(qids) == len(set(qids)), "duplicate qid in test_set.json"
for q in test_set:
    missing = [g for g in q["gold_chunks"] if g not in chunks_by_id]
    assert not missing, f"{q['qid']}: gold chunk ids not in chunks.json: {missing}"

n_ans = sum(q["answerable"] for q in test_set)
print(f"Config : {CONFIG_NAME} -> {CONFIGS[CONFIG_NAME]}")
print(f"Chunks : {len(chunks)}")
print(f"Test   : {len(test_set)} questions ({n_ans} answerable, {len(test_set) - n_ans} not answerable)")
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU will be very slow)")

## 4. Build the pipeline for this config only

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()     # peak memory then covers loading + running this config only

retriever, reranker, llm = build(CONFIGS[CONFIG_NAME], chunks)
print("Retriever:", type(retriever).__name__, "| Reranker:", type(reranker).__name__ if reranker else None,
      "| LLM:", type(llm).__name__)

## 5. Run all test questions

In [ ]:
retrieval_results, answer_results, retrieval_secs = [], [], []

for item in tqdm(test_set, desc=CONFIG_NAME):
    qid, question = item["qid"], item["question"]

    # retrieval (with or without reranker), timed separately
    t_ret = time.time()
    if reranker is not None:
        hits = retriever.search(question, CANDIDATES)
        results = reranker.rerank(question, hits, TOP_K)
    else:
        hits = retriever.search(question, TOP_K)
        results = [{"rank": i + 1, "chunk_id": h["chunk_id"], "score": h["score"]}
                   for i, h in enumerate(hits)]
    retrieval_secs.append(time.time() - t_ret)
    retrieval_results.append({"qid": qid, "results": results})

    # generation (timed)
    top_chunks = [chunks_by_id[r["chunk_id"]] for r in results]
    start = time.time()
    out = llm.generate(question, top_chunks)
    seconds = time.time() - start
    answer_results.append({"qid": qid, "answer": out["answer"], "cited_chunks": out["cited_chunks"],
                           "refused": out["refused"], "seconds": round(seconds, 4)})

peak_gpu_mb = int(torch.cuda.max_memory_allocated() // (1024 * 1024)) if torch.cuda.is_available() else 0
print(f"Done. {len(answer_results)} questions | peak GPU memory: {peak_gpu_mb} MB")

## 6. Save the result files

In [ ]:
results_dir = DEMO / "results"
results_dir.mkdir(exist_ok=True)

save_json(results_dir / f"{CONFIG_NAME}_retrieval.json", retrieval_results)
save_json(results_dir / f"{CONFIG_NAME}_answers.json", answer_results)
save_json(results_dir / f"{CONFIG_NAME}_cost.json", {"peak_gpu_mb": peak_gpu_mb})
# extra file (not part of the Task 5 spec): retrieval vs generation time split
save_json(results_dir / f"{CONFIG_NAME}_timing.json", {
    "avg_retrieval_seconds": round(float(np.mean(retrieval_secs)), 4),
    "avg_generation_seconds": round(float(np.mean([a["seconds"] for a in answer_results])), 4)})
print("Saved to", results_dir)

## 7. Task 7: metrics

Definitions follow `Task7-Evaluation.md`:
- **recall@5, mrr**: answerable questions only.
- **citation_acc**: answerable and not refused. A missing citation counts as wrong.
- **refusal_acc**: all questions.

In [ ]:
T = {q["qid"]: q for q in test_set}
RET = {r["qid"]: [x["chunk_id"] for x in r["results"]][:5] for r in retrieval_results}
ANS = {a["qid"]: a for a in answer_results}

def citation_ok(qid):
    cited, gold = ANS[qid]["cited_chunks"], set(T[qid]["gold_chunks"])
    return bool(cited) and set(cited) <= gold

def compute_metrics():
    answerable = [q for q in T if T[q]["answerable"]]
    recall, mrr = [], []
    for q in answerable:
        gold, top = set(T[q]["gold_chunks"]), RET[q]
        recall.append(len(gold & set(top)) / len(gold))
        ranks = [i for i, c in enumerate(top, 1) if c in gold]
        mrr.append(1 / ranks[0] if ranks else 0.0)

    cit_pool = [q for q in answerable if not ANS[q]["refused"]]
    return {
        "recall@5": np.mean(recall),
        "mrr": np.mean(mrr),
        "citation_acc": np.mean([citation_ok(q) for q in cit_pool]) if cit_pool else float("nan"),
        "refusal_acc": np.mean([ANS[q]["refused"] == (not T[q]["answerable"]) for q in T]),
        "avg_seconds": np.mean([a["seconds"] for a in answer_results]),
        "peak_gpu_mb": peak_gpu_mb,
    }

metrics = {k: (round(float(v), 3) if k != "peak_gpu_mb" else int(v)) for k, v in compute_metrics().items()}
pd.DataFrame([{"config": CONFIG_NAME, **metrics}]).style.hide(axis="index")

## 8. Task 7: output files (`metrics.csv`, manual check, failures)

In [ ]:
# metrics.csv: one row per config; re-running a config replaces its row
mpath = results_dir / "metrics.csv"
row = pd.DataFrame([{"config": CONFIG_NAME, **metrics}])
old = pd.read_csv(mpath) if mpath.exists() else pd.DataFrame()
if len(old):
    old = old[old["config"] != CONFIG_NAME]
pd.concat([old, row], ignore_index=True).to_csv(mpath, index=False)

# manual_check_{config}.csv: 20 answerable questions, seed 42; the team fills correct / faithful with 1 or 0
answerable_ids = sorted(q for q in T if T[q]["answerable"])
picked = random.Random(SEED).sample(answerable_ids, min(20, len(answerable_ids)))
pd.DataFrame([{"qid": q, "question": T[q]["question"], "answer": ANS[q]["answer"],
               "reference_answer": T[q]["reference_answer"], "correct": "", "faithful": ""}
              for q in picked]).to_csv(results_dir / f"manual_check_{CONFIG_NAME}.csv",
                                       index=False, encoding="utf-8-sig")

# failures_{config}.csv: no gold chunk in top 5, or wrong citation, or wrong refusal
fail_rows = []
for q, t in T.items():
    gold, a = set(t["gold_chunks"]), ANS[q]
    no_gold = t["answerable"] and not (gold & set(RET[q]))
    bad_cite = t["answerable"] and not a["refused"] and not citation_ok(q)
    bad_refusal = a["refused"] != (not t["answerable"])
    if no_gold or bad_cite or bad_refusal:
        fail_rows.append({"qid": q, "question": t["question"], "gold_chunks": t["gold_chunks"],
                          "retrieved_top5": RET[q], "cited_chunks": a["cited_chunks"], "refused": a["refused"]})
failures = pd.DataFrame(fail_rows, columns=["qid", "question", "gold_chunks", "retrieved_top5", "cited_chunks", "refused"])
failures.to_csv(results_dir / f"failures_{CONFIG_NAME}.csv", index=False, encoding="utf-8-sig")

print(f"metrics.csv updated | manual_check: {len(picked)} questions | failures: {len(failures)}")
print("\nAll configs so far:")
display(pd.read_csv(mpath))

## 9. Quick look at failures and sample answers

In [ ]:
display(failures.head(10))

print("\nSample answers:")
for q in random.Random(SEED).sample(list(T), min(3, len(T))):
    print(f"--- {q} | answerable={T[q]['answerable']} | refused={ANS[q]['refused']}")
    print("Q:", T[q]["question"])
    print("A:", ANS[q]["answer"])
    print("Cited:", ANS[q]["cited_chunks"], "| Gold:", T[q]["gold_chunks"])

## 10. Hand check (required by Task 7)
Pick 3 questions from the sample above and verify each metric by hand against the definitions. Use the cell below to see the per-question numbers.

In [ ]:
check_ids = random.Random(SEED).sample([q for q in T if T[q]["answerable"]], min(3, n_ans))
for q in check_ids:
    gold, top = set(T[q]["gold_chunks"]), RET[q]
    ranks = [i for i, c in enumerate(top, 1) if c in gold]
    print(q, "| gold:", sorted(gold), "| top5:", top)
    print("   recall@5 =", round(len(gold & set(top)) / len(gold), 3),
          "| rr =", round(1 / ranks[0], 3) if ranks else 0,
          "| refused =", ANS[q]["refused"], "| citation_ok =", citation_ok(q))

## 11. After the team fills `manual_check_*.csv`
Fill `correct` and `faithful` with 1 or 0 in each file, upload them to `results/`, then run this cell. It adds `manual_correct` and `manual_faithful` to `metrics.csv`.

In [ ]:
mdf = pd.read_csv(results_dir / "metrics.csv")
for cfg in mdf["config"]:
    f = results_dir / f"manual_check_{cfg}.csv"
    if not f.exists():
        continue
    m = pd.read_csv(f)
    if m["correct"].notna().all() and m["faithful"].notna().all() and len(m):
        mdf.loc[mdf["config"] == cfg, "manual_correct"] = round(m["correct"].astype(float).mean(), 3)
        mdf.loc[mdf["config"] == cfg, "manual_faithful"] = round(m["faithful"].astype(float).mean(), 3)
    else:
        print(f"{cfg}: manual_check not fully filled yet, skipped")
mdf.to_csv(results_dir / "metrics.csv", index=False)
display(mdf)

## 12. Download results

In [ ]:
import shutil
zip_base = str(DEMO / f"results_{CONFIG_NAME}")
wanted = [f"{CONFIG_NAME}_retrieval.json", f"{CONFIG_NAME}_answers.json", f"{CONFIG_NAME}_cost.json",
          f"{CONFIG_NAME}_timing.json", f"manual_check_{CONFIG_NAME}.csv", f"failures_{CONFIG_NAME}.csv", "metrics.csv"]
files = [results_dir / w for w in wanted if (results_dir / w).exists()]   # exact names only
tmp = DEMO / f"_pack_{CONFIG_NAME}"
shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
for f in files:
    shutil.copy(f, tmp)
zip_path = shutil.make_archive(zip_base, "zip", tmp)
shutil.rmtree(tmp)
print("Zipped:", zip_path)

if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
else:
    print("On Kaggle: the results folder and this zip are saved in the notebook output.")